# Notebook 2 — Model Zoo (multi-class breed → binary)

Train six CNN backbones with frozen ImageNet weights on the 36-breed
classification task. Per-model metrics are reported in **both**
flavours:

* **Multi-class** (top-1 breed accuracy) — what the model is optimising
* **Binary** (restricted vs unrestricted) — what the project actually needs

The binary probability is computed by summing softmax probabilities
over the restricted breed indices: `p_restricted = sum(softmax[RESTRICTED_IDX])`.


In [ ]:
# ── Mount Google Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# ── Verify GPU ────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('No GPU — go to Runtime -> Change runtime type -> T4 GPU')


In [ ]:
# ── Project root on Drive ─────────────────────────────────
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/MSc_Capstone')
for d in ['data', 'pretrained/checkpoints', 'pretrained/logs',
          'pretrained/curves', 'pretrained/inference', 'report']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

DATA   = ROOT / 'data'
CKPT   = ROOT / 'pretrained' / 'checkpoints'
LOGS   = ROOT / 'pretrained' / 'logs'
CURVES = ROOT / 'pretrained' / 'curves'
INFER  = ROOT / 'pretrained' / 'inference'
REPORT = ROOT / 'report'

print(f'Project root: {ROOT}')
print(f'Data folder:  {DATA}')


## 2.1 — Configuration and manifest


In [ ]:
from tensorflow.keras import layers
import numpy as np, pandas as pd, json, random
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import confusion_matrix, roc_auc_score

SEED = 58
tf.random.set_seed(SEED); np.random.seed(SEED); random.seed(SEED)

BATCH_SIZE    = 16
EPOCHS        = 22
PATIENCE      = 3
LEARNING_RATE = 1e-4

print('Config set.')


In [ ]:
# ── Load breed manifest written by Notebook 1 ────────────────
import json, numpy as np

with open(DATA / 'breed_to_restricted.json') as f:
    MANIFEST = json.load(f)

SELECTED_BREEDS = MANIFEST['breeds']
RESTRICTED_SET  = set(MANIFEST['restricted'])
NUM_CLASSES     = len(SELECTED_BREEDS)
BREED_TO_IDX    = {b: i for i, b in enumerate(SELECTED_BREEDS)}

# Boolean mask aligned with class index order used by image_dataset_from_directory
# (alphabetical). image_dataset_from_directory sorts class names alphabetically,
# so we mirror that ordering here.
CLASS_NAMES     = sorted(SELECTED_BREEDS)
RESTRICTED_MASK = np.array([(b in RESTRICTED_SET) for b in CLASS_NAMES], dtype=bool)
RESTRICTED_IDX  = np.where(RESTRICTED_MASK)[0]

print(f'{NUM_CLASSES} classes loaded.')
print(f'Restricted indices (in alphabetical order): {RESTRICTED_IDX.tolist()}')


## 2.2 — Dataset loader (multi-class, with class weights)


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def create_datasets(input_size=(224, 224)):
    augmentation = tf.keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomBrightness(0.1),
        layers.RandomContrast(0.1),
    ], name='augmentation')

    train_ds = tf.keras.utils.image_dataset_from_directory(
        str(DATA / 'train'), image_size=input_size, batch_size=BATCH_SIZE,
        label_mode='int', class_names=CLASS_NAMES,
        shuffle=True, seed=SEED)
    val_ds = tf.keras.utils.image_dataset_from_directory(
        str(DATA / 'val'), image_size=input_size, batch_size=BATCH_SIZE,
        label_mode='int', class_names=CLASS_NAMES, shuffle=False)
    test_ds = tf.keras.utils.image_dataset_from_directory(
        str(DATA / 'test'), image_size=input_size, batch_size=BATCH_SIZE,
        label_mode='int', class_names=CLASS_NAMES, shuffle=False)

    # Class weights for the multi-class loss (handles breed imbalance).
    y_train = np.concatenate([y.numpy() for _, y in train_ds])
    cw = compute_class_weight('balanced',
                              classes=np.arange(NUM_CLASSES), y=y_train)
    class_weight = {i: float(w) for i, w in enumerate(cw)}

    train_ds = train_ds.map(lambda x, y: (augmentation(x, training=True), y))
    AUTOTUNE = tf.data.AUTOTUNE
    return (
        train_ds.cache().shuffle(1000).prefetch(AUTOTUNE),
        val_ds.cache().prefetch(AUTOTUNE),
        test_ds.cache().prefetch(AUTOTUNE),
        class_weight,
    )

print('Dataset loader ready.')


## 2.3 — Model registry


In [ ]:
from tensorflow.keras.applications import (
    vgg16, resnet50, inception_v3, xception,
    inception_resnet_v2, nasnet
)

MODEL_ZOO = {
    'VGG16':             (vgg16.VGG16,                          vgg16.preprocess_input,              (224, 224)),
    'ResNet50':          (resnet50.ResNet50,                     resnet50.preprocess_input,           (224, 224)),
    'InceptionV3':       (inception_v3.InceptionV3,              inception_v3.preprocess_input,       (299, 299)),
    'Xception':          (xception.Xception,                     xception.preprocess_input,           (299, 299)),
    'InceptionResNetV2': (inception_resnet_v2.InceptionResNetV2, inception_resnet_v2.preprocess_input,(299, 299)),
    'NASNetMobile':      (nasnet.NASNetMobile,                   nasnet.preprocess_input,             (224, 224)),
}

def build_model(name):
    constructor, preprocess_fn, input_size = MODEL_ZOO[name]
    base = constructor(include_top=False, weights='imagenet',
                       input_shape=(*input_size, 3))
    base.trainable = False
    inputs = tf.keras.Input(shape=(*input_size, 3))
    x = preprocess_fn(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name='top1'),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5'),
        ],
    )
    total  = model.count_params()
    frozen = sum(tf.size(w).numpy() for w in base.non_trainable_weights)
    print(f'  {name}: {total:,} params ({frozen:,} frozen, {total-frozen:,} trainable)')
    return model

print('Models available:', list(MODEL_ZOO.keys()))


## 2.4 — Train all models


In [ ]:
class CSVLogger(tf.keras.callbacks.Callback):
    def __init__(self, name):
        super().__init__()
        self.path = LOGS / f'{name}_training_log.csv'
        self.rows = []
    def on_epoch_end(self, epoch, logs=None):
        self.rows.append({'epoch': epoch+1, **(logs or {})})
        pd.DataFrame(self.rows).to_csv(self.path, index=False)

def get_callbacks(name):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=PATIENCE,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            str(CKPT / f'{name}_best.keras'),
            monitor='val_loss', save_best_only=True, verbose=1),
        CSVLogger(name),
    ]

TO_RUN = ['VGG16', 'ResNet50', 'InceptionV3', 'Xception',
          'InceptionResNetV2', 'NASNetMobile']

all_results = []
all_histories = {}

for name in TO_RUN:
    print(f'\n{"="*60}\n  Training {name}\n{"="*60}')

    _, _, input_size = MODEL_ZOO[name]
    train_ds, val_ds, test_ds, class_weight = create_datasets(input_size)
    model = build_model(name)

    history = model.fit(
        train_ds, epochs=EPOCHS, validation_data=val_ds,
        callbacks=get_callbacks(name), class_weight=class_weight,
        verbose=1,
    )
    all_histories[name] = history.history

    # ---- Multi-class test metrics ----
    test_results = model.evaluate(test_ds, verbose=0)
    metrics = dict(zip(model.metrics_names, test_results))

    # ---- Binary test metrics (collapse softmax → restricted prob) ----
    y_true_mc = np.concatenate([y.numpy() for _, y in test_ds])
    y_prob_mc = model.predict(test_ds, verbose=0)
    y_prob_bin = y_prob_mc[:, RESTRICTED_IDX].sum(axis=1)
    y_true_bin = RESTRICTED_MASK[y_true_mc].astype(int)
    y_pred_bin = (y_prob_bin > 0.5).astype(int)

    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    TP, TN, FP, FN = cm[1,1], cm[0,0], cm[0,1], cm[1,0]
    metrics['bin_accuracy']  = (TP+TN) / max(TP+TN+FP+FN, 1)
    metrics['bin_precision'] = TP / max(TP+FP, 1)
    metrics['bin_recall']    = TP / max(TP+FN, 1)
    metrics['bin_f1']        = 2*TP / max(2*TP+FP+FN, 1)
    metrics['bin_auc']       = roc_auc_score(y_true_bin, y_prob_bin)
    metrics['model']         = name
    metrics['epochs_trained']= len(history.history['loss'])
    all_results.append(metrics)
    print(f'  {name}: top1={metrics["top1"]:.3f}  '
          f'bin_acc={metrics["bin_accuracy"]:.3f}  '
          f'bin_f1={metrics["bin_f1"]:.3f}  '
          f'bin_auc={metrics["bin_auc"]:.3f}')

    del model
    tf.keras.backend.clear_session()

pd.DataFrame(all_results).to_csv(LOGS / 'MASTER_results.csv', index=False)
with open(LOGS / 'MASTER_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
np.save(LOGS / 'training_histories.npy', all_histories, allow_pickle=True)

print('\nAll models trained.')


## 2.5 — Learning curves


In [ ]:
for name, hist in all_histories.items():
    fig, ax = plt.subplots(figsize=(10, 6))
    epochs = range(1, len(hist['top1']) + 1)
    ax.plot(epochs, hist['top1'],     '#1f77b4', ls='-',  lw=2, label='Train top-1')
    ax.plot(epochs, hist['val_top1'], '#1f77b4', ls='--', lw=2, label='Val top-1')
    ax.plot(epochs, hist['loss'],     '#ff7f0e', ls='-',  lw=2, label='Train loss')
    ax.plot(epochs, hist['val_loss'], '#ff7f0e', ls='--', lw=2, label='Val loss')
    ax.set_title(f'{name} — Learning Curves', fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Value')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.savefig(str(CURVES / f'{name}_learning_curves.png'), dpi=300, facecolor='white')
    plt.show(); plt.close()


## 2.6 — Confusion matrices (binary view) and metrics comparison


In [ ]:
metrics_list = []

for name in TO_RUN:
    _, _, input_size = MODEL_ZOO[name]
    test_ds = tf.keras.utils.image_dataset_from_directory(
        str(DATA / 'test'), image_size=input_size, batch_size=BATCH_SIZE,
        label_mode='int', class_names=CLASS_NAMES, shuffle=False)
    model_path = CKPT / f'{name}_best.keras'
    if not model_path.exists(): continue
    model = tf.keras.models.load_model(str(model_path))

    y_true_mc = np.concatenate([y.numpy() for _, y in test_ds])
    y_prob_mc = model.predict(test_ds, verbose=0)
    y_prob_bin = y_prob_mc[:, RESTRICTED_IDX].sum(axis=1)
    y_true_bin = RESTRICTED_MASK[y_true_mc].astype(int)
    y_pred_bin = (y_prob_bin > 0.5).astype(int)

    top1 = float(np.mean(y_prob_mc.argmax(axis=1) == y_true_mc))
    cm   = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1])
    TP, TN, FP, FN = cm[1,1], cm[0,0], cm[0,1], cm[1,0]
    acc  = (TP+TN) / max(TP+TN+FP+FN, 1)
    prec = TP / max(TP+FP, 1)
    rec  = TP / max(TP+FN, 1)
    f1   = 2*TP / max(2*TP+FP+FN, 1)
    auc_v = roc_auc_score(y_true_bin, y_prob_bin)
    metrics_list.append({'model': name, 'top1_breed': top1,
                         'accuracy': acc, 'precision': prec,
                         'recall': rec, 'f1': f1, 'auc': auc_v})

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Unrestricted','Restricted'],
                yticklabels=['Unrestricted','Restricted'])
    ax.set_title(f'{name} (binary view)', fontsize=14, fontweight='bold')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(str(INFER / f'{name}_confusion.png'), dpi=300, facecolor='white')
    plt.show(); plt.close()
    print(f'{name}: top1={top1:.3f}  Acc={acc:.3f}  P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}  AUC={auc_v:.3f}')

    del model
    tf.keras.backend.clear_session()

metrics_df = pd.DataFrame(metrics_list)
metrics_df.to_csv(INFER / 'all_models_metrics.csv', index=False)
print('\n', metrics_df.round(4).to_string(index=False))
print('\nNotebook 2 complete. Open Notebook 3 and Run all.')
